# HiggsFree: free text-to-video on Google Colab

This notebook turns a written prompt into a short MP4 with the open CogVideoX-2B model. Run each cell in order. The first run downloads about 14 GB and may be slow.

Before starting, select **Runtime → Change runtime type → T4 GPU**. Free Colab GPU access is limited and not guaranteed.

In [ ]:
# Check that Colab assigned an NVIDIA GPU.
import subprocess, torch
if not torch.cuda.is_available():
    raise RuntimeError("No GPU found. Select Runtime > Change runtime type > T4 GPU, reconnect, and run again.")
gpu_name = torch.cuda.get_device_name(0)
gpu_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"Ready: {gpu_name} with {gpu_gb:.1f} GB VRAM")
subprocess.run(["nvidia-smi"], check=False)

In [ ]:
# Install the current Hugging Face video-generation stack.
%pip install -q -U diffusers transformers accelerate sentencepiece imageio-ffmpeg

In [ ]:
#@title Describe your video
prompt = "A friendly golden retriever wearing sunglasses rides in a red convertible along the Las Vegas Strip at sunset, cinematic tracking shot, realistic lighting, smooth motion" #@param {type:"string"}
negative_prompt = "blurry, distorted, deformed, flickering, text, watermark, low quality" #@param {type:"string"}
steps = 30 #@param {type:"slider", min:20, max:50, step:5}
guidance = 6.0 #@param {type:"slider", min:1.0, max:10.0, step:0.5}
seed = 42 #@param {type:"integer"}
output_name = "higgsfree_video.mp4" #@param {type:"string"}
if not output_name.lower().endswith(".mp4"):
    output_name += ".mp4"
print("Prompt ready:", prompt)

In [ ]:
# Load the lower-memory model. The first run downloads model weights.
import gc, torch
from diffusers import CogVideoXPipeline

MODEL_ID = "zai-org/CogVideoX-2b"
torch.cuda.empty_cache()
gc.collect()
pipe = CogVideoXPipeline.from_pretrained(MODEL_ID, torch_dtype=torch.float16)
pipe.enable_sequential_cpu_offload()
pipe.vae.enable_tiling()
pipe.vae.enable_slicing()
print("Model loaded. The generation cell can take several minutes.")

In [ ]:
# Generate about six seconds of video and save it as an MP4.
from diffusers.utils import export_to_video
from IPython.display import Video, display

generator = torch.Generator(device="cpu").manual_seed(seed)
try:
    frames = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        num_frames=49,
        num_inference_steps=steps,
        guidance_scale=guidance,
        generator=generator,
    ).frames[0]
except torch.OutOfMemoryError as error:
    torch.cuda.empty_cache()
    raise RuntimeError("Colab ran out of GPU memory. Restart the runtime and make sure no other models are loaded.") from error

export_to_video(frames, output_name, fps=8)
print(f"Finished: {output_name}")
display(Video(output_name, embed=True))

In [ ]:
# Download the finished MP4 to your computer.
from google.colab import files
files.download(output_name)

## Better prompts

Describe the subject, action, location, camera movement, lighting, and visual style. Example: `A confident barber finishes a sharp fade in a modern Las Vegas barbershop, slow dolly-in, warm cinematic lighting, realistic commercial, smooth natural motion.`

Change the seed for a different result. Increase steps for quality, but expect a longer generation. Do not use a real person's face or voice without clear permission.